## Unity Catalog Setup with Objects and Privileges
###### Prerequisites : create groups in your admin console

In [0]:
-- The catalog, with its managed storage location
CREATE CATALOG IF NOT EXISTS fifa_worldcup_data
  MANAGED LOCATION 'abfss://staging@<storage>.dfs.core.windows.net/uc/fifa_platform';

-- Medallion layers
CREATE SCHEMA IF NOT EXISTS fifa_worldcup_data.staging
LOCATION 'abfss://staging@<storage>.dfs.core.windows.net/fifa_worldcup'
  COMMENT 'Fichiers bruts CSV déposés par ADF (UC1) pour la lecture Auto Loader'; -- staging will hold volume (storage credential + external location)

CREATE SCHEMA IF NOT EXISTS fifa_worldcup_data.bronze_jfjelstul;
CREATE SCHEMA IF NOT EXISTS fifa_worldcup_data.bronze_mominullptr;
CREATE SCHEMA IF NOT EXISTS fifa_worldcup_data.silver;
CREATE SCHEMA IF NOT EXISTS fifa_worldcup_data.gold;

-- Groups and their privileges

GRANT USE SCHEMA, SELECT ON SCHEMA fifa_worldcup_data.staging TO `grp-fifa-data-engineers`;
GRANT READ VOLUME  ON VOLUME fifa_worldcup_data.staging.staging_fifa_data TO `grp-fifa-data-engineers`;

GRANT USE CATALOG ON CATALOG fifa_worldcup_data TO `grp-fifa-engineers`;
GRANT USE SCHEMA, CREATE TABLE, SELECT, MODIFY
  ON SCHEMA fifa_worldcup_data.bronze TO `grp-fifa-engineers`;
GRANT USE SCHEMA, CREATE TABLE, SELECT, MODIFY
  ON SCHEMA fifa_worldcup_data.silver TO `grp-fifa-engineers`;
GRANT USE SCHEMA, CREATE TABLE, SELECT, MODIFY
  ON SCHEMA fifa_worldcup_data.gold   TO `grp-fifa-engineers`;

-- Analysts see only gold : read-only
GRANT USE CATALOG ON CATALOG fifa_worldcup_data TO `grp-fifa-analysts`;
GRANT USE SCHEMA, SELECT ON SCHEMA fifa_worldcup_data.gold TO `grp-fifa-analysts`;